In [ ]:
# ============================================================
# PIPELINE: Dataset Emosi Bahasa Indonesia (5000 data)
# Kelas: love, anger, sadness, joy, fear
# ============================================================

# ── Install dependencies (jalankan sekali di terminal/notebook) ──
# !pip install datasets pandas

import re
import pandas as pd
from datasets import load_dataset

# ============================================================
# KONFIGURASI
# ============================================================
TARGET_LABELS     = ["love", "anger", "sadness", "joy", "fear"]
SAMPLES_PER_CLASS = 1000
RANDOM_SEED       = 42
OUTPUT_FILE       = "dataset_emosi_5000.csv"

# Mapping label integer → string (khusus dair-ai/emotion)
# label: sadness(0), joy(1), love(2), anger(3), fear(4), surprise(5)
# "surprise" tidak di-map → otomatis dibuang
LABEL_MAP = {
    "sadness":  "sadness",
    "joy":      "joy",
    "love":     "love",
    "anger":    "anger",
    "fear":     "fear",
}

# ============================================================
# 1. LOAD DATASET
# ============================================================
def load_data() -> pd.DataFrame:
    """
    Load dataset emosi dari HuggingFace: dair-ai/emotion
    Sumber umum untuk emotion classification, mudah diakses tanpa login.
    Gabungkan split train + validation + test untuk memaksimalkan jumlah data.
    """
    print("⏳ Loading dataset dari HuggingFace (dair-ai/emotion)...")
    ds = load_dataset("dair-ai/emotion", split="train+validation+test")
    df = ds.to_pandas()

    # Konversi integer label → string
    int_to_str = {0: "sadness", 1: "joy", 2: "love",
                  3: "anger",   4: "fear", 5: "surprise"}
    df["label"] = df["label"].map(int_to_str)

    print(f"✅ Dataset dimuat: {len(df):,} baris")
    print("\n📊 Distribusi label AWAL:")
    print(df["label"].value_counts().to_string())
    return df

# ============================================================
# 2. FILTER LABEL
# ============================================================
def filter_labels(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pertahankan hanya label target (love/anger/sadness/joy/fear).
    Label lain (surprise) dibuang via LABEL_MAP.
    """
    df = df.copy()
    df["label"] = df["label"].map(LABEL_MAP)   # rename / NaN jika tidak ada di map
    df = df.dropna(subset=["label"])            # hapus baris dengan label NaN
    df = df[df["label"].isin(TARGET_LABELS)]    # filter ketat
    print(f"\n✅ Setelah filtering: {len(df):,} baris ({len(TARGET_LABELS)} kelas)")
    return df


# ============================================================
# 3. CLEANING TEKS
# ============================================================
def clean_text(text: str) -> str:
    """
    Preprocessing teks:
    - Lowercase
    - Hapus URL
    - Hapus mention (@username)
    - Hapus hashtag (#tag)
    - Hapus angka
    - Hapus karakter non-huruf (selain spasi)
    - Hapus spasi berlebih
    """
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", "", text)   # hapus URL
    text = re.sub(r"@\w+", "", text)                # hapus mention
    text = re.sub(r"#\w+", "", text)                # hapus hashtag
    text = re.sub(r"\d+", "", text)                 # hapus angka
    text = re.sub(r"[^a-z\s]", "", text)            # hapus non-huruf (simpan spasi)
    text = re.sub(r"\s+", " ", text).strip()        # hapus spasi berlebih
    return text


def apply_cleaning(df: pd.DataFrame) -> pd.DataFrame:
    """Terapkan clean_text ke seluruh DataFrame, lalu hapus duplikat & kosong."""
    df = df.copy()
    df["clean_text"] = df["text"].apply(clean_text)

    before = len(df)
    df = df.drop_duplicates(subset=["clean_text"])          # hapus duplikat
    df = df[df["clean_text"].str.strip() != ""]             # hapus teks kosong
    df = df.dropna(subset=["text", "clean_text", "label"])  # hapus null
    after = len(df)

    print(f"\n🧹 Cleaning selesai: {before:,} → {after:,} baris "
          f"(dihapus {before - after:,} duplikat/kosong)")
    return df


# ============================================================
# 4. BALANCING
# ============================================================
def balance_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ambil SAMPLES_PER_CLASS sampel acak per kelas (random_state=RANDOM_SEED).
    Jika suatu kelas kurang dari SAMPLES_PER_CLASS, ambil semua & beri peringatan.
    Hasil dikocok ulang agar tidak berurutan per kelas.
    """
    parts = []
    print(f"\n⚖️  Balancing dataset ({SAMPLES_PER_CLASS} per kelas):")
    for label in TARGET_LABELS:
        subset = df[df["label"] == label]
        n = min(len(subset), SAMPLES_PER_CLASS)
        if n < SAMPLES_PER_CLASS:
            print(f"  ⚠️  {label:<10}: hanya {n} data tersedia (kurang dari {SAMPLES_PER_CLASS})")
        else:
            print(f"  ✅ {label:<10}: {n} data diambil")
        sampled = subset.sample(n=n, random_state=RANDOM_SEED)
        parts.append(sampled)

    balanced = (pd.concat(parts)
                  .sample(frac=1, random_state=RANDOM_SEED)
                  .reset_index(drop=True))
    return balanced


# ============================================================
# 5. BUILD FINAL DATAFRAME
# ============================================================
def build_final_df(df: pd.DataFrame) -> pd.DataFrame:
    """Susun kolom final: text | clean_text | label."""
    final = df[["text", "clean_text", "label"]].copy().reset_index(drop=True)
    return final


# ============================================================
# 6. SIMPAN & LAPORAN
# ============================================================
def save_and_report(df: pd.DataFrame) -> None:
    """Simpan ke CSV dan tampilkan ringkasan akhir."""
    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print(f"\n{'='*52}")
    print(f"  💾 Dataset disimpan ke: {OUTPUT_FILE}")
    print(f"{'='*52}")
    print(f"\n📊 RINGKASAN DATASET FINAL")
    print(f"  Total data   : {len(df):,}")
    print(f"  Jumlah kelas : {df['label'].nunique()}")
    print(f"\n  Distribusi per label:")
    dist = df["label"].value_counts().sort_index()
    for label, count in dist.items():
        bar = "█" * (count // 50)
        print(f"    {label:<10} : {count:>5}  {bar}")
    print(f"\n  Contoh data (3 baris pertama):")
    print(df[["text", "clean_text", "label"]].head(3).to_string(index=False))


# ============================================================
# MAIN PIPELINE
# ============================================================
def run_pipeline() -> pd.DataFrame:
    print("=" * 52)
    print("  PIPELINE DATASET EMOSI — 5000 DATA")
    print("=" * 52)

    df_raw      = load_data()
    df_filtered = filter_labels(df_raw)
    df_cleaned  = apply_cleaning(df_filtered)
    df_balanced = balance_dataset(df_cleaned)
    df_final    = build_final_df(df_balanced)
    save_and_report(df_final)

    return df_final


# ── Jalankan ────────────────────────────────────────────────
if __name__ == "__main__":
    df = run_pipeline()

  PIPELINE DATASET EMOSI — 5000 DATA
⏳ Loading dataset dari HuggingFace (dair-ai/emotion)...


README.md: 0.00B [00:00, ?B/s]

c:\Users\Lenovo\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\datasets--dair-ai--emotion. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back 

train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


...

test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

✅ Dataset dimuat: 20,000 baris

📊 Distribusi label AWAL:
label
joy         6761
sadness     5797
anger       2709
fear        2373
love        1641
surprise     719

✅ Setelah filtering: 19,281 baris (5 kelas)

🧹 Cleaning selesai: 19,281 → 19,239 baris (dihapus 42 duplikat/kosong)

⚖️  Balancing dataset (1000 per kelas):
  ✅ love      : 1000 data diambil
  ✅ anger     : 1000 data diambil
  ✅ sadness   : 1000 data diambil
  ✅ joy       : 1000 data diambil
  ✅ fear      : 1000 data diambil

  💾 Dataset disimpan ke: dataset_emosi_5000.csv

📊 RINGKASAN DATASET FINAL
  Total data   : 5,000
  Jumlah kelas : 5

  Distribusi per label:
    anger      :  1000  ████████████████████
    fear       :  1000  ████████████████████
    joy        :  1000  ████████████████████
    love       :  1000  ████████████████████
    sadness    :  1000  ████████████████████

  Contoh data (3 baris pertama):
                                                                                                         